In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp04/ex_1.py ──
"""
Shared infrastructure for MLFP04 Exercise 1 — Clustering Zoo.

Contains: customer-feature loading & standardisation, metric helpers,
subsampling utilities, and output-directory management.

Technique-specific code (K-means elbow, linkage methods, DBSCAN epsilon
search, HDBSCAN, spectral Laplacian, AutoMLEngine) does NOT belong
here — it lives in the per-technique files under modules/mlfp04/solutions/ex_1/.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex1_clustering"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared random state so every technique file is reproducible
RANDOM_STATE: int = 42


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore e-commerce customers
# ════════════════════════════════════════════════════════════════════════


def load_customers() -> tuple[pl.DataFrame, list[str]]:
    """Load the e-commerce customer dataset and return (df, numeric_feature_cols).

    The dataset (from MLFP03) is ~6K rows of Singapore e-commerce customers
    with recency, frequency, monetary, basket-size, and channel features.
    Clustering is unsupervised segmentation: no labels, just behaviour.
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")
    numeric_types = (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in numeric_types and c not in ("customer_id",)
    ]
    return customers.drop_nulls(subset=feature_cols), feature_cols


def standardise(
    df: pl.DataFrame, feature_cols: list[str]
) -> tuple[np.ndarray, StandardScaler]:
    """Return (X_scaled, scaler). Zero mean, unit variance — mandatory for
    all distance-based clustering."""
    X, _, _ = to_sklearn_input(df, feature_columns=feature_cols)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler


# ════════════════════════════════════════════════════════════════════════
# SUBSAMPLING — spectral / hierarchical are O(n^2) or worse
# ════════════════════════════════════════════════════════════════════════


def subsample(
    X: np.ndarray, n: int, seed: int = RANDOM_STATE
) -> tuple[np.ndarray, np.ndarray]:
    """Return (X_sub, idx) where idx are the original row indices chosen."""
    rng = np.random.default_rng(seed)
    n = min(n, X.shape[0])
    idx = rng.choice(X.shape[0], n, replace=False)
    return X[idx], idx


# ════════════════════════════════════════════════════════════════════════
# METRICS
# ════════════════════════════════════════════════════════════════════════


def score_partition(X: np.ndarray, labels: np.ndarray) -> dict[str, float]:
    """Compute silhouette, Calinski-Harabasz, Davies-Bouldin.

    Points with label == -1 (DBSCAN noise) are excluded. Returns NaN
    fields if fewer than 2 valid clusters remain.
    """
    valid = labels != -1
    labs = labels[valid]
    data = X[valid]
    if data.shape[0] < 2 or len(set(labs.tolist())) < 2:
        return {
            "n_clusters": len(set(labs.tolist())),
            "silhouette": float("nan"),
            "calinski_harabasz": float("nan"),
            "davies_bouldin": float("nan"),
        }
    return {
        "n_clusters": len(set(labs.tolist())),
        "silhouette": float(silhouette_score(data, labs)),
        "calinski_harabasz": float(calinski_harabasz_score(data, labs)),
        "davies_bouldin": float(davies_bouldin_score(data, labs)),
    }


def agreement(labels_a: np.ndarray, labels_b: np.ndarray) -> dict[str, float]:
    """ARI and NMI between two label vectors, ignoring noise (-1)."""
    valid = (labels_a >= 0) & (labels_b >= 0)
    if valid.sum() < 2:
        return {"ari": float("nan"), "nmi": float("nan")}
    return {
        "ari": float(adjusted_rand_score(labels_a[valid], labels_b[valid])),
        "nmi": float(normalized_mutual_info_score(labels_a[valid], labels_b[valid])),
    }


def print_metric_row(name: str, m: dict[str, Any]) -> None:
    """One-line summary of a partition's metrics."""
    print(
        f"  {name:<14} K={m['n_clusters']:>3}  "
        f"sil={m['silhouette']:>7.4f}  "
        f"CH={m['calinski_harabasz']:>9.0f}  "
        f"DB={m['davies_bouldin']:>7.4f}"
    )


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION OUTPUT PATH
# ════════════════════════════════════════════════════════════════════════


def out_path(filename: str) -> Path:
    """Return a path under OUTPUT_DIR for a visualisation artefact."""
    return OUTPUT_DIR / filename




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 1.2: Hierarchical Clustering with Four Linkage Methods
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build an agglomerative hierarchy bottom-up via the linkage matrix
#   - Compare single, complete, average, and Ward's linkage behaviours
#   - Read a dendrogram and cut it at a chosen K
#   - Choose linkage based on cluster shape expectations
#
# PREREQUISITES: 01_kmeans.py.
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — agglomerative merging and what linkage means
#   2. Build — fit four linkage methods on a subsample
#   3. Train — score each linkage partition against the others
#   4. Visualise — the Ward dendrogram
#   5. Apply — Singapore NTUC FairPrice store-cluster taxonomy
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

import numpy as np
from dotenv import load_dotenv
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)


load_dotenv()



## THEORY — Agglomerative Merging and Linkage


In [ ]:
# Each linkage method defines "distance between clusters" differently:
#   single   = min pairwise distance → elongated chains
#   complete = max pairwise distance → compact spheres
#   average  = mean pairwise distance
#   ward     = increase in within-cluster variance → K-means-like
# Ward's is the production default.



## TASK 2 — BUILD: Subsample and fit four linkage methods


Fit one linkage method and score it at the CUT_K cut.


In [ ]:
customers, feature_cols = load_customers()
X_scaled, _ = standardise(customers, feature_cols)
n_samples = X_scaled.shape[0]

# TODO: Subsample 2000 rows using the shared subsample() helper.
# Hint: X_hier, idx_hier = subsample(X_scaled, n=____, seed=RANDOM_STATE)
X_hier, idx_hier = ____
n_hier = X_hier.shape[0]

CUT_K = 5
LINKAGE_METHODS = ["single", "complete", "average", "ward"]

print("=" * 70)
print("  Hierarchical Clustering on Singapore E-commerce Customers")
print("=" * 70)
print(f"  Subsample: {n_hier:,} of {n_samples:,} customers  cut K={CUT_K}")
print(
    f"\n  {'Linkage':<10} {'K':>4} {'Silhouette':>12} {'CH':>10} {'DB':>8} {'Time':>8}"
)
print("  " + "─" * 55)


def fit_linkage(method: str) -> dict:
    t0 = time.perf_counter()
    # TODO: Compute the scipy linkage matrix Z. Use metric="euclidean".
    # Hint: Z = linkage(X_hier, method=____, metric="euclidean")
    Z = ____
    elapsed = time.perf_counter() - t0

    # TODO: Cut the dendrogram to get CUT_K clusters using
    # fcluster(Z, t=CUT_K, criterion="maxclust"). Subtract 1 for 0-based labels.
    labels = ____

    k_actual = len(set(labels.tolist()))
    if k_actual >= 2:
        sil = silhouette_score(X_hier, labels)
        ch = calinski_harabasz_score(X_hier, labels)
        db = davies_bouldin_score(X_hier, labels)
    else:
        sil, ch, db = -1.0, 0.0, float("inf")
    return {
        "Z": Z,
        "labels": labels,
        "n_clusters": k_actual,
        "silhouette": sil,
        "ch": ch,
        "db": db,
        "time": elapsed,
    }


hier_results = {m: fit_linkage(m) for m in LINKAGE_METHODS}
for m, r in hier_results.items():
    print(
        f"  {m:<10} {r['n_clusters']:>4} {r['silhouette']:>12.4f} "
        f"{r['ch']:>10.0f} {r['db']:>8.4f} {r['time']:>7.2f}s"
    )



### Checkpoint 1


In [ ]:
assert len(hier_results) == 4, "Task 2: all four linkage methods required"
assert all("silhouette" in r for r in hier_results.values()), "Task 2: scoring gap"
print("\n  [ok] Checkpoint 1 passed — four linkage methods fitted\n")



## TASK 3 — TRAIN: Which linkage wins?


In [ ]:
# TODO: Find the linkage method with the HIGHEST silhouette. Use
# max(..., key=lambda x: x[1]["silhouette"]) on hier_results.items().
best_method = ____
print(
    f"  Best linkage by silhouette: {best_method[0]} "
    f"(sil={best_method[1]['silhouette']:.4f})"
)

ward_sil = hier_results["ward"]["silhouette"]
single_sil = hier_results["single"]["silhouette"]
print(f"  Ward's silhouette: {ward_sil:.4f}")
print(
    f"  Single silhouette: {single_sil:.4f}  "
    f"({'chains' if single_sil < ward_sil - 0.05 else 'competitive'})"
)



### Checkpoint 2


In [ ]:
assert best_method[0] in LINKAGE_METHODS, "Task 3: best method invalid"
assert hier_results["ward"]["n_clusters"] == CUT_K, "Task 3: Ward cut mismatch"
print("\n  [ok] Checkpoint 2 passed — Ward's linkage partition extracted\n")



## TASK 4 — VISUALISE: Ward dendrogram


In [ ]:
try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(12, 6))
    dendrogram(
        hier_results["ward"]["Z"],
        truncate_mode="lastp",
        p=30,
        ax=ax,
        leaf_font_size=9,
        color_threshold=0.7 * hier_results["ward"]["Z"][-CUT_K, 2],
    )
    ax.set_title(f"Ward Dendrogram — cut at K={CUT_K}")
    ax.set_xlabel("Cluster size (leaves)")
    ax.set_ylabel("Merge distance")
    fig.tight_layout()
    fig.savefig(str(out_path("02_hier_ward_dendrogram.png")), dpi=120)
    plt.close(fig)
    print(f"  Saved: {out_path('02_hier_ward_dendrogram.png')}")
except ImportError as e:  # pragma: no cover
    raise ImportError("02_hierarchical.py requires matplotlib") from e

print("\n  Read the dendrogram: bar height = merge distance; long vertical")
print("  gaps above a cut mean that cut is a robust, 'natural' K.")



### Checkpoint 3


In [ ]:
assert out_path("02_hier_ward_dendrogram.png").exists(), "Task 4: dendrogram missing"
print("\n  [ok] Checkpoint 3 passed — dendrogram rendered\n")



## TASK 5 — APPLY: NTUC FairPrice Store-Cluster Taxonomy


In [ ]:
# SCENARIO: FairPrice's ~230 stores get a tree-structured taxonomy.
# Merchandising thinks in trees; give them a dendrogram. Different K
# values serve different decisions (4 for national, 12 for regional).
#
# BUSINESS IMPACT: ~S$128M / year trade-promotion spend. Data-driven
# clustering recovers ~10% waste = S$12.8M / year.

print("  APPLY — NTUC FairPrice Store Taxonomy")
print("  ─────────────────────────────────────────────────────────────────")
ward_labels = hier_results["ward"]["labels"]

# TODO: Compute sizes = np.bincount(ward_labels). Print each cluster's
# size and percentage of n_hier.
sizes = ____
for i, n in enumerate(sizes):
    print(f"    Ward cluster {i}: {n:>5,} customers ({n/n_hier:6.1%})")
print("    Estimated annual promo waste recovery: S$12.8M.")



### Checkpoint 4


In [ ]:
assert int(sizes.sum()) == n_hier, "Task 5: Ward partition size mismatch"
assert len(sizes) == CUT_K, "Task 5: Ward cut should yield CUT_K clusters"
print("\n  [ok] Checkpoint 4 passed — Ward taxonomy valid\n")



## REFLECTION


[x] Agglomerative merging builds a dendrogram bottom-up
  [x] Four linkage methods produce different cluster shapes
  [x] Read a dendrogram: height = merge distance; cut = partition
  [x] Ward's is the production default for compact clusters
  [x] Mapped the tree onto an NTUC FairPrice store taxonomy — S$12.8M/yr

  KEY INSIGHT: When the business thinks in a TREE, give them a tree.

  Next: 03_density_based.py — clusters of arbitrary SHAPE.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

